In [0]:
%sql
use catalog ext_cat;

In [0]:
%sql
drop table if exists orders;

In [0]:
landing_zone =  '/Volumes/ext_cat/default/raw'
orders_data = landing_zone + '/ordershistory'
checkpoint_path =  landing_zone + '/orders_checkpoint'

In [0]:
orders_data

'/Volumes/ext_cat/default/raw/ordershistory'

In [0]:
checkpoint_path 

'/Volumes/ext_cat/default/raw/orders_checkpoint'

streaming data

In [0]:
# spark.readStream.format("cloudFiles") \
#   .option("cloudFiles.format", "csv") \
#   .option("cloudFiles.schemaLocation", checkpoint_path) \
#   .option("cloudFiles.schemaHints", "order_id integer, order_date string, customer_id integer, order_status string") \
#   .option("cloudFiles.maxFilesPerTrigger", 1) \
#   .option("cloudFiles.useNotifications", "true") \
#   .option(
#     "cloudFiles.schemaEvolutionMode",
#     "addNewColumns"
#           )
#   .load(orders_data) \
#   .writeStream \
#   .option("checkpointLocation", checkpoint_path) \
#   .trigger(once=True) \
#   .toTable("_sqldf")

In [0]:
ordersdf= spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation", checkpoint_path) \
  .option("cloudFiles.inferSchema", "true") \
  .option("cloudFiles.inferColumnTypes", 'true') \
  .load(orders_data) \
  

In [0]:
%sql
drop table if exists ext_cat.default.orders;

Place a file in volume otherwise read/write will fail

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

Due to trigger(availableNow=True), streaming just ran once, and processed all file in the source and ended

In [0]:
%sql
select * from orders

order_id,order_date,customer_id,order_status,_rescued_data
1001,2026-08-25,201,Delivered,null
1002,2026-08-26,205,Shipped,null
1003,2026-08-27,202,Processing,null
1004,2026-08-28,208,Cancelled,null
1005,2026-08-30,201,Delivered,null
1006,2026-09-01,210,Shipped,null
1007,2026-09-02,204,Processing,null
1008,2026-09-03,207,Pending,null
1009,2026-09-05,203,Pending,null


Place anothr file in vol and write again

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

In [0]:
%sql
select * from orders

order_id,order_date,customer_id,order_status,_rescued_data
1001,2026-08-25,201,Delivered,null
1002,2026-08-26,205,Shipped,null
1003,2026-08-27,202,Processing,null
1004,2026-08-28,208,Cancelled,null
1005,2026-08-30,201,Delivered,null
1006,2026-09-01,210,Shipped,null
1007,2026-09-02,204,Processing,null
1008,2026-09-03,207,Pending,null
1009,2026-09-05,203,Pending,null
10101,2025-08-25,2101,Delivered,null


#### - Add another file with a new column and try. 
#### - First it will fail with error - `org.apache.spark.sql.catalyst.util.UnknownFieldException: [UNKNOWN_FIELD_EXCEPTION.NEW_FIELDS_IN_FILE] Encountered unknown fields during parsing: [new_col], which can be fixed by an automatic retry: true`
#### - Then we need to retry the readstream once, and then retry writestream for it to work

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

In [0]:
%sql
select * from orders

order_id,order_date,customer_id,order_status,_rescued_data,new_col
10101,null,null,Delivered,"{""order_date"":""25-08-2025"",""customer_id"":""ABC"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",1
10102,null,2105,Shipped,"{""order_date"":""26-08-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",2
10103,null,2102,Processing,"{""order_date"":""27-08-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",3
10104,null,2108,Cancelled,"{""order_date"":""28-08-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",4
10105,null,2101,Delivered,"{""order_date"":""30-08-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",5
10106,null,2110,Shipped,"{""order_date"":""01-09-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",6
10107,null,2104,Processing,"{""order_date"":""02-09-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",7
10108,null,2107,Pending,"{""order_date"":""03-09-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",8
10109,null,1203,Pending,"{""order_date"":""05-09-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",1
1001,2026-08-25,201,Delivered,null,null


The new file had a different date format and also a new column . New column got added after retry. Date format mismatched rows were loaded to rescued_data col

In [0]:
%fs
ls /Volumes/ext_cat/default/raw/orders_checkpoint

path,name,size,modificationTime
dbfs:/Volumes/ext_cat/default/raw/orders_checkpoint/__tmp_path_dir/,__tmp_path_dir/,0,1788628124000
dbfs:/Volumes/ext_cat/default/raw/orders_checkpoint/_schemas/,_schemas/,0,1788628113000
dbfs:/Volumes/ext_cat/default/raw/orders_checkpoint/commits/,commits/,0,1788628126000
dbfs:/Volumes/ext_cat/default/raw/orders_checkpoint/metadata,metadata,45,1788628124000
dbfs:/Volumes/ext_cat/default/raw/orders_checkpoint/offsets/,offsets/,0,1788628126000
dbfs:/Volumes/ext_cat/default/raw/orders_checkpoint/sources/,sources/,0,1788628126000


**RocksDB** is a fast key value store. This is where metadata is persisted in checkpoint location for tracking ingestion progress

In [0]:
 %fs
ls /Volumes/ext_cat/default/raw/orders_checkpoint/sources/0/rocksdb/

path,name,size,modificationTime
dbfs:/Volumes/ext_cat/default/raw/orders_checkpoint/sources/0/rocksdb/0.zip,0.zip,172,1788628126000
dbfs:/Volumes/ext_cat/default/raw/orders_checkpoint/sources/0/rocksdb/1.zip,1.zip,3734,1788628128000
dbfs:/Volumes/ext_cat/default/raw/orders_checkpoint/sources/0/rocksdb/2.zip,2.zip,3859,1788628284000
dbfs:/Volumes/ext_cat/default/raw/orders_checkpoint/sources/0/rocksdb/3.zip,3.zip,3859,1788628287000
dbfs:/Volumes/ext_cat/default/raw/orders_checkpoint/sources/0/rocksdb/4.zip,4.zip,3953,1788628458000
dbfs:/Volumes/ext_cat/default/raw/orders_checkpoint/sources/0/rocksdb/5.zip,5.zip,3935,1788628622000
dbfs:/Volumes/ext_cat/default/raw/orders_checkpoint/sources/0/rocksdb/SSTs/,SSTs/,0,1788628128000
dbfs:/Volumes/ext_cat/default/raw/orders_checkpoint/sources/0/rocksdb/__tmp_path_dir/,__tmp_path_dir/,0,1788628126000
dbfs:/Volumes/ext_cat/default/raw/orders_checkpoint/sources/0/rocksdb/logs/,logs/,0,1788628128000
